# Notebook 05 — Explainability Analysis

**PharmaSentinel** | Harshita Adlakha

Three XAI analyses:
1. Integrated Gradients token attributions
2. Attention Rollout per layer
3. Global feature importance via TF-IDF + SHAP for the SVM baseline

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from transformers import AutoTokenizer

from pharmasentinel.models import PharmaSentinelMTL
from pharmasentinel.explainability import (
    PharmaSentinelExplainer, extract_attention, attention_rollout, top_k_tokens
)
from pharmasentinel.data import load_raw_data, preprocess
from pharmasentinel.utils import set_seed

set_seed(42)
CHECKPOINT = 'distilbert-base-uncased'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
raw = load_raw_data('../data/drugsComTrain_raw.tsv', '../data/drugsComTest_raw.tsv')
df, cond_enc, _ = preprocess(raw, top_k_conditions=50)
NUM_CONDITIONS = df['condition'].nunique()

model = PharmaSentinelMTL(CHECKPOINT, num_conditions=NUM_CONDITIONS)
ckpt  = torch.load('../checkpoints/distilbert_mtl/best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
explainer  = PharmaSentinelExplainer(model, tokenizer, device=DEVICE)

print('Model and explainer ready.')

## 1. Integrated Gradients — Token Attribution Heatmap

In [ ]:
def plot_attribution(tokens, scores, title, ax, max_tokens=30):
    """Render a coloured token bar chart on the given axis."""
    special = {'[CLS]', '[SEP]', '[PAD]'}
    pairs   = [(t, s) for t, s in zip(tokens[:max_tokens], scores[:max_tokens]) if t not in special]
    toks, vals = zip(*pairs) if pairs else ([], [])

    colors  = plt.cm.RdYlGn(np.array(vals))
    bars    = ax.barh(range(len(toks)), vals, color=colors)
    ax.set_yticks(range(len(toks)))
    ax.set_yticklabels(toks, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Attribution Score')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlim(0, 1.05)


examples = [
    ("This drug completely changed my life for the better. My depression lifted after three weeks and I finally feel like myself again. The side effects were minimal.", 'Positive'),
    ("Absolutely terrible. Gained 15 pounds, constant headaches, and my mood was horrible. Stopped after 4 weeks. Would not recommend to anyone.", 'Negative'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

for ax, (text, label) in zip(axes, examples):
    tokens, scores = explainer.integrated_gradients(text, task='sentiment', n_steps=50)
    plot_attribution(tokens, scores, f'{label} Review Attribution', ax)

plt.suptitle('Integrated Gradients: Sentiment Task', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/ig_attributions.png', bbox_inches='tight')
plt.show()

## 2. Attention Rollout

In [ ]:
review = 'This medication helped my anxiety tremendously but caused severe insomnia at night.'

tokens, layers = extract_attention(model, tokenizer, review, device=DEVICE)
rollout_scores = attention_rollout(layers)

# Plot token scores
fig, ax = plt.subplots(figsize=(10, 3))
clean   = [t for t in tokens if t not in {'[CLS]', '[SEP]', '[PAD]'}]
cscores = rollout_scores[1: len(clean) + 1]

ax.bar(range(len(clean)), cscores, color=plt.cm.Blues(cscores / cscores.max()))
ax.set_xticks(range(len(clean)))
ax.set_xticklabels(clean, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Attention Rollout Score')
ax.set_title('Attention Rollout — CLS → Token Flow', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/attention_rollout.png', bbox_inches='tight')
plt.show()

print('Top-5 tokens by attention rollout:')
for tok, score in top_k_tokens(tokens, rollout_scores, k=5):
    print(f'  {tok:20s} → {score:.4f}')

## 3. Global Feature Importance (TF-IDF SVM + SHAP)

In [ ]:
import shap
from pharmasentinel.data import build_splits
from pharmasentinel.models import svm_baseline

train_df, _, test_df = build_splits(df)
X_train = train_df['review_clean'].tolist()[:5000]    # subset for speed
y_train = train_df['sentiment'].values[:5000]
X_test  = test_df['review_clean'].tolist()[:500]

svm_pipe = svm_baseline()
svm_pipe.fit(X_train, y_train)

# SHAP for linear SVM via feature pipeline
tfidf    = svm_pipe.named_steps['tfidf']
clf      = svm_pipe.named_steps['clf']
X_test_tfidf = tfidf.transform(X_test)

explainer_shap = shap.LinearExplainer(clf, X_test_tfidf, feature_dependence='independent')
shap_values    = explainer_shap.shap_values(X_test_tfidf)

# Plot top features for Positive class (class index 2)
shap.summary_plot(
    shap_values[2], X_test_tfidf,
    feature_names=tfidf.get_feature_names_out(),
    max_display=20, plot_type='bar', show=False
)
plt.title('SHAP Feature Importance — Positive Sentiment (SVM)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/shap_global.png', bbox_inches='tight')
plt.show()